# Part 4: Production-Grade Modeling & Leakage Remediation
**Course:** AIML Capstone Project  
**Task:** Engineering untainted temporal features, training optimized classifier, and evaluating realistic performance bounds.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

print("Production environment initialized.")

## 1. Implementing Temporal Cutoff to Remediate Target Leakage

In [ ]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
tickets = pd.read_csv("support_tickets.csv")
labels = pd.read_csv("churn_labels.csv")

target_col = [col for col in labels.columns if 'churn' in col.lower()][0]

orders['order_date'] = pd.to_datetime(orders['order_date'])
tickets['ticket_date'] = pd.to_datetime(tickets['ticket_date'])
snapshot_date = pd.to_datetime("2025-09-30")

historical_orders = orders[orders['order_date'] <= snapshot_date]
historical_tickets = tickets[tickets['ticket_date'] <= snapshot_date]

order_features = historical_orders.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'gross_amount': 'sum'
}).reset_index()
order_features.columns = ['customer_id', 'recency', 'frequency', 'monetary']

ticket_features = historical_tickets.groupby('customer_id').size().reset_index(name='complaints_count')

df = pd.merge(customers, order_features, on='customer_id', how='left')
df = pd.merge(df, ticket_features, on='customer_id', how='left')
df = pd.merge(df, labels[['customer_id', target_col]], on='customer_id', how='inner')

df['recency'] = df['recency'].fillna(365)
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
df['complaints_count'] = df['complaints_count'].fillna(0)

print(f"Untainted feature matrix dimensions: {df.shape}")

## 2. Stratified Train-Validation Splitting

In [ ]:
X = df[['recency', 'frequency', 'monetary', 'complaints_count']]
y = df[target_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")

## 3. Training Regularized Production Classifier

In [ ]:
model = RandomForestClassifier(n_estimators=150, random_state=42, max_depth=6, min_samples_leaf=4)
model.fit(X_train, y_train)

preds = model.predict(X_val)
probs = model.predict_proba(X_val)[:, 1]

print(f"Overall Accuracy: {accuracy_score(y_val, preds):.4f}")
print(f"ROC-AUC Performance Score: {roc_auc_score(y_val, probs):.4f}")
print("\nClassification Matrix Report:")
print(classification_report(y_val, preds))

## 4. Final Overfitting Verification Check

In [ ]:
train_preds = model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)
val_acc = accuracy_score(y_val, preds)

print(f"Training split performance accuracy: {train_acc:.4f}")
print(f"Validation split performance accuracy: {val_acc:.4f}")
print(f"Observed performance gap delta: {abs(train_acc - val_acc):.4f}")